## Bimodal distribution via `NoisyValue.minimum` and `NoisyValue.maximum`

This example builds two `NoisyValue` objects with noise around different centers.
Then it computes:
- `lo = left.minimum(right)`
- `hi = left.maximum(right)`

Finally, it draws samples from both and randomly mixes them.
Because `lo` mostly contributes left-side values and `hi` mostly contributes right-side values,
the mixed result is **bimodal**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sympy.stats import Normal

from main import NoisyValue

# Reproducible randomness for mixing
rng = np.random.default_rng(7)
n = 20000

# Two noisy values centered apart
left = NoisyValue.from_distribution(
    -2.0,
    Normal,
    0,
    0.7,
    provenance='left',
    name_prefix='L',
)
right = NoisyValue.from_distribution(
    2.0,
    Normal,
    0,
    0.7,
    provenance='right',
    name_prefix='R',
)

# Use class methods requested in this notebook
lo = left.minimum(right)
hi = left.maximum(right)

# Sample each branch
lo_samples = lo.sample_n(n, seed=101)
hi_samples = hi.sample_n(n, seed=202)

# Randomly choose either minimum or maximum draw at each index
choose_hi = rng.random(n) < 0.5
bimodal = np.where(choose_hi, hi_samples, lo_samples)

# Visualize
bins = np.linspace(-5, 5, 80)
plt.figure(figsize=(9, 5))
plt.hist(lo_samples, bins=bins, alpha=0.35, density=True, label='left.minimum(right)')
plt.hist(hi_samples, bins=bins, alpha=0.35, density=True, label='left.maximum(right)')
plt.hist(bimodal, bins=bins, alpha=0.85, density=True, label='bimodal mix')
plt.title('Bimodal Distribution from NoisyValue.minimum / maximum')
plt.xlabel('value')
plt.ylabel('density')
plt.legend()
plt.show()

# Quick numeric check: center density should be below side peaks
hist, edges = np.histogram(bimodal, bins=80, range=(-5, 5), density=True)
centers = (edges[:-1] + edges[1:]) / 2
center_density = hist[np.argmin(np.abs(centers - 0.0))]
left_peak = hist[np.argmin(np.abs(centers + 2.0))]
right_peak = hist[np.argmin(np.abs(centers - 2.0))]
print(f'center density near 0: {center_density:.3f}')
print(f'left/right peak densities near -2/+2: {left_peak:.3f}, {right_peak:.3f}')

TypeError: Normal() got an unexpected keyword argument 'dist_args'